# Lesson 05 - groupby 與 KPI 彙總

1. `groupby` 是 pandas 分析的核心。
2. 這一章用它計算訂單數、營收、平均客單價等 KPI。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR_CANDIDATES = [
    Path("../data/raw"),
    Path("data/raw"),
    Path("/content/Py_dataAna/code/data/raw"),
    Path("/content/code/data/raw"),
]

DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if (path / "orders.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "找不到 CSV 資料。請確認 code/data/raw/ 內有 orders.csv 等資料檔，"
        "在 Colab 可先上傳整個專案資料夾或掛載 Google Drive。"
    )

print("Using data folder:", DATA_DIR.resolve())

Using data folder: E:\py_20260620\data\raw


In [2]:
customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
sessions = pd.read_csv(DATA_DIR / "sessions.csv")
events = pd.read_csv(DATA_DIR / "events.csv")
ab_assignments = pd.read_csv(DATA_DIR / "ab_assignments.csv")

tables = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "sessions": sessions,
    "events": events,
    "ab_assignments": ab_assignments,
}
pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": len(df.columns)} for name, df in tables.items()]
)

,table,rows,columns
0,customers,2500,5
1,products,60,3
2,orders,22000,5
3,order_items,39627,5
4,sessions,70000,7
5,events,232067,6
6,ab_assignments,2500,3


## 建立訂單營收表

In [ ]:
items = order_items.copy()
items["line_revenue"] = items["quantity"] * items["unit_price"] * (1 - items["discount_rate"])
# 統計各訂單的成交金額
order_revenue = items.groupby("order_id", as_index=False)["line_revenue"].sum()
order_facts = order_revenue.merge(orders, on="order_id", how="left")
print(order_facts)
# query 條件
completed = order_facts.query("status == 'completed'")
completed.head()

       order_id  line_revenue  customer_id  order_date     status payment_type
0             1        538.65         2323  2025-05-02  completed       wallet
1             2      4,357.70          117  2024-07-14  completed       wallet
2             3      3,536.00          160  2025-03-29  cancelled         card
3             4      1,193.40         1240  2024-07-08  completed          atm
4             5     13,175.50          714  2024-08-12  completed          atm
...         ...           ...          ...         ...        ...          ...
21995     21996      6,198.25         2390  2025-08-06  completed         card
21996     21997      3,160.00         2467  2025-10-04  completed         card
21997     21998        951.90         1072  2024-07-05   refunded          cod
21998     21999      2,204.00         1977  2025-03-31  completed         card
21999     22000      5,771.20          843  2024-09-06  completed         card

[22000 rows x 6 columns]


,order_id,line_revenue,customer_id,order_date,status,payment_type
0,1,538.65,2323,2025-05-02,completed,wallet
1,2,"4,357.70",117,2024-07-14,completed,wallet
3,4,"1,193.40",1240,2024-07-08,completed,atm
4,5,"13,175.50",714,2024-08-12,completed,atm
5,6,"3,657.50",2005,2025-01-14,completed,card


## 計算整體 KPI

In [4]:
total_revenue = completed["line_revenue"].sum()
completed_orders = completed["order_id"].nunique()
aov = total_revenue / completed_orders

print("完成訂單數:", completed_orders)
print("成交金額總數:", round(total_revenue, 2))
print("AOV平均客單金額:", round(aov, 2))

完成訂單數: 20413
成交金額總數: 101771799.75
AOV平均客單金額: 4985.64


## 依付款方式彙總

In [5]:
completed.groupby("payment_type").agg(
    orders=("order_id", "nunique"),
    revenue=("line_revenue", "sum"),
    aov=("line_revenue", "mean"),
).sort_values("revenue", ascending=False)#依"revenue"排序, 由大到小

,orders,revenue,aov
payment_type,,,
card,10166,"50,567,692.45","4,974.20"
atm,5111,"25,452,594.55","4,979.96"
wallet,3076,"15,343,032.20","4,987.98"
cod,2060,"10,408,480.55","5,052.66"


## 小練習

把 `payment_method` 改成其他類別欄位，觀察 KPI 差異。